#### This code PLOTS THE FIGURES needed for the paper 

## Functions

In [1]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

In [2]:
from matplotlib.image import NonUniformImage
import matplotlib.pyplot as plt

In [3]:
H1_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/matriz1_100nt"
directory= os.scandir(H1_folder)
list_of_files=get_directory(H1_folder)

newlist2=[]
for item in list_of_files:
    newlist2.append(H1_folder+'/'+str(item))
#print(newlist)

H1=[]

for j in np.arange(0,len(newlist2)):
#for j in np.arange(0,2):
    h1=np.loadtxt(newlist2[j])
    H1.append(h1)
    
    
H2_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/matriz2_100nt"
directory= os.scandir(H2_folder)
list_of_files=get_directory(H2_folder)

newlist3=[]
for item in list_of_files:
    newlist3.append(H2_folder+'/'+str(item))
#print(newlist)

H2=[]

for j in np.arange(0,len(newlist3)):
#for j in np.arange(0,2):
    h2=np.loadtxt(newlist3[j])
    H2.append(h2)   
    
    
H3_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/matrix3_coordinates_plotpequeno"
directory= os.scandir(H3_folder)
list_of_files=get_directory(H3_folder)

newlist4=[]
for item in list_of_files:
    newlist4.append(H3_folder+'/'+str(item))
#print(newlist)

H3=[]

for j in np.arange(0,len(newlist4)):
#for j in np.arange(0,2):
    h3=np.loadtxt(newlist4[j])
    H3.append(h3)       

In [4]:
print(len(H1))

320


### Matrix per month

In [5]:
zeros = np.zeros(shape=(100,100))    
for i in np.arange(0,len(H1)):
    zeros=np.add(zeros,H1[i])

zeros2 = np.zeros(shape=(100,100))    
for j in np.arange(0,len(H2)):
    zeros2=np.add(zeros2,H2[j]) 
    
zeros3 = np.zeros(shape=(100,100))    
for k in np.arange(0,len(H3)):
    zeros3=np.add(zeros3,H3[k])     
    
print("Addition of two matrix") 
print(zeros)

print("Addition of two matrix") 
print(zeros2)

print("Addition of two matrix") 
print(zeros3)

Addition of two matrix
[[ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 ...
 [ 0.  0.  0. ...  0.  0. 60.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0. 55.]]
Addition of two matrix
[[ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 ...
 [ 0.  0.  0. ... 10.  6.  2.]
 [ 0.  0.  0. ...  1.  3.  0.]
 [ 0.  0.  0. ... 12.  2. 11.]]
Addition of two matrix
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


## Plot #1 Cometo centric distance vs outgassing

In [37]:
#MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events9"
MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events10"

#Importing data
#titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance']
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmoduluspeaks', 'cometo-centric distance','x','roots','bmodulusaverage','y','z']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

#print(data3)
data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['gas_production_rates'][j])==True or pd.isnull(data3_copy['cometo-centric distance'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)
        
data3_copy=data3_copy.reset_index(drop=True)

import math

#print(data3_copy)
x=data3_copy['gas_production_rates']
xx=x.astype(float)
y=data3_copy['cometo-centric distance']
yy=y.astype(float)

#print(xx)
xxx=[]

for i in np.arange(0,len(xx)):
    if np.isnan(xx[i])==True:
        xxx.append(np.nan)
    else:
        xxx.append(math.log(xx[i],10))
        
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=500        

In [7]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy   

### Amount of MM waves

In [38]:
HHH, xedges, yedges= np.histogram2d(xxx,yy, bins=(xedges,yedges), range=[[xmin,xmax],[ymin,ymax]])

'With nan values instead of 0'
HHH_copy=HHH.copy()
for i in np.arange(0,len(HHH_copy[0])):
    for j in np.arange(0,len(HHH_copy[0])):
        if HHH_copy[i][j]==0:
            HHH_copy[i][j]=np.nan

extent = [xmin,xmax, ymin, ymax]
#plt.figure()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(HHH_copy.T ,origin='lower', extent=extent, aspect='auto')
fig, ax= plt.subplots()

im=ax.imshow(HHH_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(im, label='# Mirror modes per bin')
#plt.clim(0,0.001) #This limit shows how to low values looks like

#ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('r (km)', fontsize=12)

Text(0, 0.5, 'r (km)')

### Amount of plasma density data

In [9]:
'With nan values instead of 0'
zeros_copy=zeros.copy()
for i in np.arange(0,len(zeros_copy[0])):
    for j in np.arange(0,len(zeros_copy[0])):
        if zeros_copy[i][j]==0:
            zeros_copy[i][j]=np.nan


#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(zeros_copy.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(zeros.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros_copy.T ,origin='lower', extent=extent, aspect='auto')
cbar=plt.colorbar(im,label='PDDs per bin')
cbar.formatter.set_powerlimits((0, 0))
im.set_clim(0,70000)
#plt.clim(0,70000) #This limit shows how to low values looks like
ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('r (km)', fontsize=12)

Text(0, 0.5, 'r (km)')

### Errors

In [10]:
#print(zeros_copy)
zeros_copy2=zeros_copy.copy()
for i in np.arange(0,len(zeros_copy)):
    for j in np.arange(0,len(zeros_copy)):
        if np.isnan(zeros_copy[i][j])==False and zeros_copy[i][j]!=0:
            zeros_copy2[i][j]=1/zeros_copy[i][j]
            
    
#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(zeros_copy.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(zeros.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros_copy2.T ,origin='lower', extent=extent, aspect='auto')
cbar=plt.colorbar(im,label='Errors per bin (1/PDDs)')
cbar.formatter.set_powerlimits((0, 0))
#im.set_clim(0,1e-2)
im.set_clim(0,1e-3)
#plt.clim(0,70000) #This limit shows how to low values looks like
ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('r (km)', fontsize=12)    
                   

Text(0, 0.5, 'r (km)')

### Amount of normalized MM waves

In [11]:
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=500
HHH, xedges, yedges= np.histogram2d(xxx,yy, bins=(xedges,yedges), range=[[xmin,xmax],[ymin,ymax]])

'With 0 values'
H_cometo_centric=matrix_division(HHH,zeros)

'With nan values instead of 0'
matrix_copy=H_cometo_centric.copy()
for i in np.arange(0,len(matrix_copy[0])):
    for j in np.arange(0,len(matrix_copy[0])):
        if matrix_copy[i][j]==0:
            matrix_copy[i][j]=np.nan

#print(matrix_copy)            
extent = [24.5,29, 0, 500]

#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')

#plt.imshow(H_cometo_centric.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')
cbar=plt.colorbar(im,label='# Normalised mirror modes per bin')
cbar.formatter.set_powerlimits((0, 2))
#plt.clim(0,0.001) #This limit shows how to low values looks like
im.set_clim(0,0.001)
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('r (km)', fontsize=12)

Text(0, 0.5, 'r (km)')

## Plot #2 Bmodulus (deltaB/B) peaks vs Outgassing NOT USEFUL

In [12]:
MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events3"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

#print(data3)
data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['gas_production_rates'][j])==True or pd.isnull(data3_copy['bmodulus'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)
        
data3_copy=data3_copy.reset_index(drop=True)

import math
#print(data3_copy)
x2=data3_copy['gas_production_rates']
xx2=x2.astype(float)
y2=data3_copy['bmodulus']
yy2=y2.astype(float)

xxx2=[]

for i in np.arange(0,len(xx2)):
    #print(i)
    if np.isnan(xx2[i])==True:
        xxx2.append(np.nan)
    
    else:
        xxx2.append(math.log(xx2[i],10))

        
xmin=24.5
xmax=29
ymin=0
ymax=100      

plt.figure()
plt.hist2d(xxx2,yy2,bins=100,range=[[xmin, xmax], [ymin, ymax]]) #, cmap=plt.cm.jet)
plt.colorbar(label='Amount of MM waves')
#plt.xscale('log')
#plt.ylim([min(yy),max(yy)]) #-5 and 10
plt.ylim([0,100])
#plt.xlim([24.5,29])
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)

Text(0, 0.5, '|B| (nT)')

In [13]:
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=100
HHH2, xedges, yedges= np.histogram2d(xxx2,yy2, bins=(xedges,yedges), range=[[xmin,xmax],[ymin,ymax]])

'With 0 values'
H_bmodulus=matrix_division(HHH2,zeros2)

'With nan values instead of 0'
matrix_copy2=H_bmodulus.copy()
for i in np.arange(0,len(matrix_copy2[0])):
    for j in np.arange(0,len(matrix_copy2[0])):
        if matrix_copy2[i][j]==0:
            matrix_copy2[i][j]=np.nan


extent = [24.5,29, 0, 100]
plt.figure()
plt.imshow(H_bmodulus.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(matrix_copy2.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(label='Amount of normalized MM waves')
#plt.clim(0,0.005) #This limit shows how to low values looks like
plt.clim(0,0.001) #This limit shows how to low values looks like
#plt.clim(0,0.335) #This limit shows how the original data looks like

plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)

Text(0, 0.5, '|B| (nT)')

In [14]:
plt.figure()
extent = [24.5,29, 0,100]

'With nan values instead of 0'
matrix_copy2_zeros=zeros2.copy()
for i in np.arange(0,len(matrix_copy2_zeros[0])):
    for j in np.arange(0,len(matrix_copy2_zeros[0])):
        if matrix_copy2_zeros[i][j]==0:
            matrix_copy2_zeros[i][j]=np.nan



plt.imshow(zeros2.T ,origin='lower',extent=extent ,aspect='auto')
#plt.imshow(matrix_copy2_zeros.T ,origin='lower',extent=extent ,aspect='auto')
#plt.clim(0,10000) #This limit shows how to low values looks like
plt.colorbar(label='Amount of LAP data')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)

Text(0, 0.5, '|B| (nT)')

## Plot #3 Bmodulus (mean values) vs Outgassing 

In [15]:
#MM_prominence_data4="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events9"
MM_prominence_data4="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events10"

#Importing data
#titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulusaverage']
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmoduluspeaks', 'cometo-centric distance','x','roots','bmodulusaverage','y','z']
dataa= pd.read_table(str(MM_prominence_data4), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

#print(data3)
data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['gas_production_rates'][j])==True or pd.isnull(data3_copy['bmodulusaverage'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)
        
data3_copy=data3_copy.reset_index(drop=True)
import math
#print(data3_copy)
x2=data3_copy['gas_production_rates']
xx2=x2.astype(float)
y2=data3_copy['bmodulusaverage']
yy2=y2.astype(float)

xxx2=[]

for i in np.arange(0,len(xx2)):
    #print(i)
    if np.isnan(xx2[i])==True:
        xxx2.append(np.nan)
    
    else:
        xxx2.append(math.log(xx2[i],10))



xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=100

### Amount of MM waves

In [16]:
HHH2, xedges, yedges= np.histogram2d(xxx2,yy2, bins=100, range=[[xmin,xmax],[ymin,ymax]])

'With nan values instead of 0'
matrix_copy2=HHH2.copy()
for i in np.arange(0,len(matrix_copy2[0])):
    for j in np.arange(0,len(matrix_copy2[0])):
        if matrix_copy2[i][j]==0:
            matrix_copy2[i][j]=np.nan


extent = [24.5,29, 0, 100]
#plt.figure()

fig, ax=plt.subplots()

#plt.imshow(HHH2.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(matrix_copy2.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(matrix_copy2.T ,origin='lower', extent=extent, aspect='auto')

#plt.colorbar(label='Amount of MM waves')
cbar=plt.colorbar(im, label='# Mirror modes per bin')
cbar.formatter.set_powerlimits((-1, 0))
#plt.clim(0,4) #This limit shows how to low values looks like
im.set_clim(0,4)
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)
#ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')



In [17]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy  

### Amount of plasma density data

In [18]:
'With nan values instead of 0'
zeros2_copy=zeros2.copy()
for i in np.arange(0,len(zeros2_copy[0])):
    for j in np.arange(0,len(zeros2_copy[0])):
        if zeros2_copy[i][j]==0:
            zeros2_copy[i][j]=np.nan


#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(zeros2.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros2_copy.T ,origin='lower', extent=extent, aspect='auto')

plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)
cbar=plt.colorbar(im, label='PDDs per bin')
cbar.formatter.set_powerlimits((0, 0))

#im.set_clim(0,4)
ax.set_facecolor('grey')

### Amount of normalized MM waves

In [19]:
xedges=100
yedges=100
xmin=24.5
xmax=29
ymin=0
ymax=100
HHH2, xedges, yedges= np.histogram2d(xxx2,yy2, bins=(xedges,yedges), range=[[xmin,xmax],[ymin,ymax]])

'With 0 values'
H_bmodulus=matrix_division(HHH2,zeros2)

'With nan values instead of 0'
matrix_copy2=H_bmodulus.copy()
for i in np.arange(0,len(matrix_copy2[0])):
    for j in np.arange(0,len(matrix_copy2[0])):
        if matrix_copy2[i][j]==0:
            matrix_copy2[i][j]=np.nan


extent = [24.5,29, 0, 100]
fig, ax=plt.subplots()

#plt.imshow(H_bmodulus.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(matrix_copy2.T ,origin='lower', extent=extent, aspect='auto')

cbar=plt.colorbar(im,label='# Normalised mirror modes per bin')
cbar.formatter.set_powerlimits((0, 0))
#plt.clim(0,0.005) #This limit shows how to low values looks like

#plt.clim(0,0.001) #This limit shows how to low values looks like
#plt.clim(0,0.0005) #This limit shows how to low values looks like
#plt.clim(0,0.335) #This limit shows how the original data looks like

plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)',fontsize=12)

im.set_clim(0,0.0005)
ax.set_facecolor('grey')

### Errors

In [20]:
#print(zeros_copy)
zeros2_copy2=zeros2_copy.copy()
for i in np.arange(0,len(zeros2_copy)):
    for j in np.arange(0,len(zeros2_copy)):
        if np.isnan(zeros2_copy[i][j])==False and zeros2_copy[i][j]!=0:
            zeros2_copy2[i][j]=1/zeros2_copy[i][j]
            
    
#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(zeros_copy.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(zeros.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros2_copy2.T ,origin='lower', extent=extent, aspect='auto')
cbar=plt.colorbar(im,label='Errors per bin (1/PDDs)')
cbar.formatter.set_powerlimits((0, 0))
#im.set_clim(0,1e-2)
im.set_clim(0,0.0005)
#plt.clim(0,70000) #This limit shows how to low values looks like
ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('|B| (nT)', fontsize=12)    
                   

Text(0, 0.5, '|B| (nT)')

## Plot #4 x vs root

In [21]:
#MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events9"
MM_prominence_data3="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events10"

#Importing data
#titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus', 'cometo-centric distance','x','roots']
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmoduluspeaks', 'cometo-centric distance','x','roots','bmodulusaverage','y','z']
dataa= pd.read_table(str(MM_prominence_data3), header=None, names=titles, sep='\t')
data3=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data3=data3.reset_index(drop=True)

#print(data3)
data3_copy=data3.copy()
#Filter nan values
for j in np.arange(0,len(data3)):
    if pd.isnull(data3_copy['x'][j])==True or pd.isnull(data3_copy['roots'][j])==True:
        data3_copy.drop(j, axis=0, inplace=True)
        
        
data3_copy=data3_copy.reset_index(drop=True)

import math

#print(data3_copy)
x=data3_copy['x']
xx=x.astype(float)
y=data3_copy['roots']
yy=y.astype(float)

### Amount of MM waves

In [22]:
xmin=-20
xmax=200
ymin=0
ymax=450
HHH, xedges, yedges= np.histogram2d(xx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])

extent = [-20,200, 0, 450]

'With nan values instead of 0'
HHH_copy=HHH.copy()
for i in np.arange(0,len(HHH_copy[0])):
    for j in np.arange(0,len(HHH_copy[0])):
        if HHH_copy[i][j]==0:
            HHH_copy[i][j]=np.nan


fig, ax= plt.subplots()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(HHH_copy.T ,origin='lower', extent=extent, aspect='auto')
plt.colorbar(im,label='# Mirror modes per bin')

#plt.clim(0,7) #This limit shows how to low values looks like
im.set_clim(0,7)
ax.set_facecolor('grey')
plt.xlabel('x (km)',fontsize=12)
plt.ylabel(r'$\rho$ (km)',fontsize=12)


Text(0, 0.5, '$\\rho$ (km)')

In [23]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy   

### Amount of plasma density data

In [24]:
'With nan values instead of 0'
zeros3_copy=zeros3.copy()
for i in np.arange(0,len(zeros3_copy[0])):
    for j in np.arange(0,len(zeros3_copy[0])):
        if zeros3_copy[i][j]==0:
            zeros3_copy[i][j]=np.nan

fig, ax= plt.subplots()
#plt.imshow(zeros3.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros3_copy.T ,origin='lower', extent=extent, aspect='auto')

cbar=plt.colorbar(im,label='PDDs per bin')
cbar.formatter.set_powerlimits((-1, 0))
#plt.clim(0,20000) #This limit shows how to low values looks like

plt.xlabel('x (km)',fontsize=12)
plt.ylabel(r'$\rho$ (km)',fontsize=12)
#plt.clim(0,7) #This limit shows how to low values looks like
im.set_clim(0,20000)
ax.set_facecolor('grey')


### Amount of normalized MM waves

In [25]:
'With 0 values'
H_roots=matrix_division(HHH,zeros3)

'With nan values instead of 0'
matrix_copy=H_roots.copy()
for i in np.arange(0,len(matrix_copy[0])):
    for j in np.arange(0,len(matrix_copy[0])):
        if matrix_copy[i][j]==0:
            matrix_copy[i][j]=np.nan

#print(matrix_copy)            
extent = [-20,200, 0, 450]


fig, ax= plt.subplots()
#plt.imshow(H_roots.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')

cbar=plt.colorbar(im,label='# Normalised mirror modes per bin')
cbar.formatter.set_powerlimits((0, 0))
#plt.clim(0,0.0004) #This limit shows how to low values looks like
im.set_clim(0,0.0004)
ax.set_facecolor('grey')
plt.xlabel('x (km)',fontsize=12)
plt.ylabel(r'$\rho$ (km)',fontsize=12)


Text(0, 0.5, '$\\rho$ (km)')

### Errors

In [26]:
#print(zeros_copy)
zeros3_copy2=zeros3_copy.copy()
for i in np.arange(0,len(zeros3_copy)):
    for j in np.arange(0,len(zeros3_copy)):
        if np.isnan(zeros3_copy[i][j])==False and zeros3_copy[i][j]!=0:
            zeros3_copy2[i][j]=1/zeros3_copy[i][j]
            
    
#plt.figure()
fig, ax=plt.subplots()
#plt.imshow(zeros_copy.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(zeros.T ,origin='lower', extent=extent, aspect='auto')
im=ax.imshow(zeros3_copy2.T ,origin='lower', extent=extent, aspect='auto')
cbar=plt.colorbar(im,label='Errors per bin (1/PDDs)')
cbar.formatter.set_powerlimits((0, 0))
#im.set_clim(0,1e-2)
im.set_clim(0,0.0004)
#plt.clim(0,70000) #This limit shows how to low values looks like
ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('x (km)',fontsize=12)
plt.ylabel(r'$\rho$ (km)', fontsize=12) 

Text(0, 0.5, '$\\rho$ (km)')

## Plot #5 z vs y

In [27]:
plt.figure()

plt.imshow(zeros3.T ,origin='lower', extent=extent, aspect='auto')



plt.colorbar(label='Amount of LAP data')
plt.clim(0,20000) #This limit shows how to low values looks like

plt.xlabel('x (km)',fontsize=12)
plt.ylabel('$\sqrt{y^{2}+z^{2}}$ (km)',fontsize=12)

Text(0, 0.5, '$\\sqrt{y^{2}+z^{2}}$ (km)')

In [28]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy   

In [29]:
MM_prominence_data4="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events9"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmoduluspeaks', 'cometo-centric distance','x','roots','bmodulusaverage','y','z']
dataa= pd.read_table(str(MM_prominence_data4), header=None, names=titles, sep='\t')
data4=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data4=data4.reset_index(drop=True)

#print(data4)
data4_copy=data4.copy()
#Filter nan values
for j in np.arange(0,len(data4)):
    if pd.isnull(data4_copy['y'][j])==True or pd.isnull(data4_copy['z'][j])==True:
        data4_copy.drop(j, axis=0, inplace=True)
        
        
data4_copy=data4_copy.reset_index(drop=True)

import math

#print(data3_copy)
x=data4_copy['z']
xx=x.astype(float)
y=data4_copy['y']
yy=y.astype(float)

print(min(xx))
print(max(xx))
print(min(yy))
print(max(yy))
xmin=-450
xmax=450
ymin=-1000
ymax=400


plt.figure()
plt.hist2d(xx,yy,bins=100, range=[[xmin,xmax],[ymin,ymax]]) #, cmap=plt.cm.jet)
plt.colorbar(label='Amount of MM waves')
#plt.xscale('log')
#plt.ylim([min(yy),500]) #-5 and 10
#plt.ylim([0,500]) #-5 and 10
plt.clim(0,7) #This limit shows how to low values looks like
plt.xlabel('z (km)')#,fontsize=10)
plt.ylabel('y (km)')

-504.46483870967734
404.0
-964.2174193548387
339.340625


Text(0, 0.5, 'y (km)')

In [30]:
def matrix_division(array1,array2):
    array1_copy=array1.copy()
    for i in np.arange(0,len(array1)):
        for j in np.arange(0,len(array1[0])):
            if array2[i][j]==0:
                array1_copy[i][j]=0
            else:    
                array1_copy[i][j]=array1[i][j]/array2[i][j]
                
    return array1_copy 

In [31]:
xmin=-450
xmax=450
ymin=-1000
ymax=400

HHH, xedges, yedges= np.histogram2d(xx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])

'With 0 values'
H_yvsz=matrix_division(HHH,zeros4)

'With nan values instead of 0'
matrix_copy=H_yvsz.copy()
for i in np.arange(0,len(matrix_copy[0])):
    for j in np.arange(0,len(matrix_copy[0])):
        if matrix_copy[i][j]==0:
            matrix_copy[i][j]=np.nan

#print(matrix_copy)            
extent = [-450,450, -1000, 400]
plt.figure()

plt.imshow(H_yvsz.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')



plt.colorbar(label='Amount of normalized MM waves')
#plt.clim(0,0.0003) #This limit shows how to low values looks like

plt.xlabel('x (km)')#,fontsize=10)
plt.ylabel('$\sqrt{y^{2}+z^{2}}$ (km)')

NameError: name 'zeros4' is not defined

In [ ]:
plt.figure()

plt.imshow(zeros4.T ,origin='lower', extent=extent, aspect='auto')



plt.colorbar(label='Amount of LAP data')
#plt.clim(0,50000) #This limit shows how to low values looks like

plt.xlabel('x (km)')#,fontsize=10)
plt.ylabel('$\sqrt{y^{2}+z^{2}}$ (km)')

## PLOT #6

In [ ]:
newoutgassingpath="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproductionwithzcoordinate.txt"

#Importing data
titles=['Timestamp', 'Neutral gas density', 'Derived gas production rate', 'date', 'z']
dataa= pd.read_table(str(newoutgassingpath), header=None, names=titles, sep='\t')
data4=dataa.iloc[np.arange(1,len(dataa)),:] #delete the first row
data4=data4.reset_index(drop=True)
data4=data4.set_index('date')
#print(data4)
#data4=data4.loc['2014-11-01 00:00:00','2016-02-19 23:59:59' ]
#data4_copy=data4.copy()
#data4=dataa.iloc[np.arange(169428,760948),:] 
print(data4)
#Filter nan values

#for j in np.arange(0,len(data4)):
 #   if pd.isnull(data4_copy['y'][j])==True or pd.isnull(data4_copy['z'][j])==True:
  #      data4_copy.drop(j, axis=0, inplace=True)
        
        
#data4_copy=data4_copy.reset_index(drop=True)

data4_copy=data4.copy()

import math

#print(data3_copy)

x=data4_copy['Derived gas production rate']
xx=x.astype(float)
y=data4_copy['z']
yy=y.astype(float)

xxx2=[]

for i in np.arange(0,len(xx)):
    #print(i)
    if np.isnan(xx[i])==True:
        xxx2.append(np.nan)
    else:
        xxx2.append(math.log(xx[i],10))

print(min(xx2))
print(max(xx2))
print(min(yy))
print(max(yy))


In [ ]:
print(xx2[0])
print(yy)

In [ ]:
xmin=24.5
xmax=29
ymin=-350
ymax=350



plt.figure()
plt.hist2d(xxx2,yy,bins=100, range=[[xmin,xmax],[ymin,ymax]]) #, cmap=plt.cm.jet)
#plt.colorbar(label='Amount of MM waves')
#plt.xscale('log')
#plt.ylim([min(yy),500]) #-5 and 10
#plt.ylim([0,500]) #-5 and 10


#plt.imshow(H_bmodulus.T ,origin='lower', extent=extent, aspect='auto')
#im=ax.imshow(matrix_copy2.T ,origin='lower', extent=extent, aspect='auto')

cbar=plt.colorbar(label='Gas production rate data')
cbar.formatter.set_powerlimits((0, 0))
#plt.clim(0,4e3) #This limit shows how to low values looks like



plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('z (km)',fontsize=12)







In [ ]:
xedges=100
yedges=100
HHH, xedges, yedges= np.histogram2d(xxx2,yy, bins=(xedges,yedges), range=[[xmin,xmax],[ymin,ymax]])

'With nan values instead of 0'
matrix_copy=HHH.copy()
for i in np.arange(0,len(matrix_copy[0])):
    for j in np.arange(0,len(matrix_copy[0])):
        if matrix_copy[i][j]==0:
            matrix_copy[i][j]=np.nan




extent = [xmin,xmax, ymin, ymax]
#plt.figure()
#plt.imshow(HHH.T ,origin='lower', extent=extent, aspect='auto')
#plt.imshow(HHH_copy.T ,origin='lower', extent=extent, aspect='auto')
fig, ax= plt.subplots()

im=ax.imshow(matrix_copy.T ,origin='lower', extent=extent, aspect='auto')

cbar=plt.colorbar(im,label='Gas production rate data')
cbar.formatter.set_powerlimits((0, 0))
#plt.clim(0,0.001) #This limit shows how to low values looks like

#ax.set_facecolor((0.8,0.8,0.8))
ax.set_facecolor('grey')
plt.xlabel('log Q ($s^{-1}$)',fontsize=12)
plt.ylabel('z (km)', fontsize=12)


#ax.set_facecolor('grey')

In [ ]:
print(HHH)